# Test Notebook: Open Library Public Domain Opportunity Radar

This notebook checks `helper_code.py` without the real dumps. Every test
builds a small input whose correct answer we can work out by hand, runs a
helper function on it, and compares the result with that answer.

The tests are grouped by what could go wrong:

1. **Rule functions (pure Python).** Year parsing, publisher names and
   the copyright rules, including every boundary year.
2. **Spark twins.** Each rule exists twice: a readable Python version and
   a Spark column expression used on the full data. We run both on the
   same inputs and require identical answers.
3. **End-to-end on a miniature dump.** We write tiny `.txt.gz` files in
   the real Open Library format (including a malformed record and a
   deprecated Wikidata value), run the whole pipeline, and check every
   number against hand calculations.
4. **Style.** `pycodestyle` on the helper module.

Run it from top to bottom (Kernel > Restart & Run All). A summary at the
end says how many tests passed. The tests write only to a temporary
folder, which is deleted afterwards.

In [1]:
from pathlib import Path
import gzip
import importlib
import json
import sys
import tempfile
import unittest

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import helper_code as helper

helper = importlib.reload(helper)

# A small local Spark session is enough for the miniature data.
spark = helper.create_spark(app_name="ol-pd-radar-tests",
                            driver_memory="2g", shuffle_partitions=4)
print("Spark", spark.version)

Spark 4.0.1


## 1 Rule functions

These tests pin the behaviour at the boundaries, where off-by-one
mistakes happen. On **1 January 2027** a US work published in 1931
reaches the end of its 95-year term (1931 + 95 = 2026, protected until
31 December 2026). In the Philippines, an author who died in 1976 has
been protected for 50 years counted from 1 January 1977, which ends on
31 December 2026.

In [2]:
class TestYearParsing(unittest.TestCase):
    # Free-text publication dates -> first plausible year.

    def test_publication_years(self):
        cases = {
            "February 27, 1997": 1997,
            "c1931": 1931,
            "1931-1932": 1931,          # the first year is kept
            "Printed 1931; reprinted 1950": 1931,
            "19--": None,               # an unknown decade is not a year
            "1200": None,               # before printing (MIN_VALID_YEAR)
            "2030": None,               # after the dump date
            "12345": None,              # five digits are not a year
            "": None,
            None: None,
        }
        for text, expected in cases.items():
            with self.subTest(text=text):
                self.assertEqual(helper.parse_year(text), expected)

    def test_life_years(self):
        # Author dates keep the LATEST year (conservative) and handle BC.
        cases = {
            "1901.7.": 1901,
            "1963)": 1963,
            "31 October 1987": 1987,
            "c. 1850 or 1851": 1851,
            "1016": 1016,
            "384 BC": -384,
            "384 B.C.": -384,
            "unknown": None,
            None: None,
        }
        for text, expected in cases.items():
            with self.subTest(text=text):
                self.assertEqual(helper.parse_life_year(text), expected)

    def test_wikidata_time(self):
        self.assertEqual(
            helper.parse_wikidata_time_year("+1931-05-03T00:00:00Z"), 1931)
        self.assertEqual(
            helper.parse_wikidata_time_year("-0384-00-00T00:00:00Z"), -384)
        self.assertIsNone(helper.parse_wikidata_time_year("garbage"))


class TestPublisherNames(unittest.TestCase):

    def test_normalization(self):
        cases = {
            "Penguin Books Ltd.": "penguin books",
            "PENGUIN BOOKS": "penguin books",
            "Hal Leonard Publishing Corporation": "hal leonard publishing",
            "Simon & Schuster, Inc.": "simon schuster",
            "  ": "",
            None: "",
        }
        for name, expected in cases.items():
            with self.subTest(name=name):
                self.assertEqual(helper.normalize_publisher(name), expected)


class TestCopyrightRules(unittest.TestCase):

    def test_us_boundaries(self):
        cases = {1500: "pd", 1930: "pd", 1931: "enters", 1932: "renewal",
                 1963: "renewal", 1964: "protected_pre78",
                 1977: "protected_pre78", 1978: "protected_post78",
                 2026: "protected_post78", None: "unknown"}
        for year, expected in cases.items():
            with self.subTest(year=year):
                self.assertEqual(helper.us_status(year), expected)

    def test_us_rule_moves_with_reference_year(self):
        # On 1 Jan 2028 the 1932 works enter and 1931 works are PD.
        self.assertEqual(helper.us_status(1932, reference_year=2028),
                         "enters")
        self.assertEqual(helper.us_status(1931, reference_year=2028), "pd")

    def test_philippine_life_plus_50(self):
        rule = helper.life_plus_status
        self.assertEqual(rule(1975, 1, 1, 50), "pd")
        self.assertEqual(rule(1976, 1, 1, 50), "enters")
        self.assertEqual(rule(1977, 1, 1, 50), "protected")
        self.assertEqual(rule(-384, 1, 1, 50), "pd")
        # two authors, only one death year known -> cannot decide
        self.assertEqual(rule(1900, 2, 1, 50), "unknown_death")
        self.assertEqual(rule(None, 1, 0, 50), "unknown_death")
        self.assertEqual(rule(None, 0, 0, 50), "unknown_author")

    def test_life_plus_70(self):
        rule = helper.life_plus_status
        self.assertEqual(rule(1955, 1, 1, 70), "pd")
        self.assertEqual(rule(1956, 1, 1, 70), "enters")
        self.assertEqual(rule(1957, 1, 1, 70), "protected")

    def test_death_year_source(self):
        self.assertEqual(helper.choose_death_year(1951, 1950),
                         (1950, "wikidata"))
        self.assertEqual(helper.choose_death_year(1951, None),
                         (1951, "open_library"))
        self.assertEqual(helper.choose_death_year(None, None), (None, None))

## 2 Spark expressions agree with the Python rules

The full-data pipeline cannot call the Python functions row by row
(that would be very slow), so it uses equivalent Spark expressions. If
the two versions ever disagreed, the report's numbers would silently
differ from the documented rules. We therefore evaluate both on the same
inputs, including every year from 1450 to 2026.

In [3]:
from pyspark.sql import functions as F


class TestSparkTwins(unittest.TestCase):

    def test_year_from_text(self):
        texts = ["February 27, 1997", "c1931", "19--", "1200", "2030",
                 "12345", "1931-1932", "Printed 1931; reprinted 1950", None]
        df = spark.createDataFrame([(t,) for t in texts], "t string")
        got = [r[0] for r in df.select(
            helper.year_from_text(F.col("t"))).collect()]
        self.assertEqual(got, [helper.parse_year(t) for t in texts])

    def test_life_year_from_text(self):
        texts = ["1901.7.", "1963)", "c. 1850 or 1851", "384 BC",
                 "384 B.C.", "unknown", "1016", None]
        df = spark.createDataFrame([(t,) for t in texts], "t string")
        got = [r[0] for r in df.select(
            helper.life_year_from_text(F.col("t"))).collect()]
        self.assertEqual(got, [helper.parse_life_year(t) for t in texts])

    def test_publisher_key(self):
        names = ["Penguin Books Ltd.", "Hal Leonard Publishing Corporation",
                 "Simon & Schuster, Inc.", "  ", "Ateneo de Manila Univ."]
        df = spark.createDataFrame([(n,) for n in names], "n string")
        got = [r[0] for r in df.select(
            helper.publisher_key(F.col("n"))).collect()]
        self.assertEqual(got, [helper.normalize_publisher(n) for n in names])

    def test_us_status_every_year(self):
        years = list(range(1450, 2027)) + [None]
        df = spark.createDataFrame([(y,) for y in years], "y int")
        got = [r[0] for r in df.select(
            helper.us_status_column(F.col("y"))).collect()]
        self.assertEqual(got, [helper.us_status(y) for y in years])

    def test_life_plus_every_case(self):
        rows = [(d, n, k) for d in [None, -384, 1900, 1955, 1956, 1957,
                                    1975, 1976, 1977, 2020]
                for n in [0, 1, 2] for k in [0, 1, 2] if k <= n]
        df = spark.createDataFrame(rows, "d int, n int, k int")
        for term in (50, 70):
            got = [r[0] for r in df.select(helper.life_plus_status_column(
                F.col("d"), F.col("n"), F.col("k"), term)).collect()]
            expected = [helper.life_plus_status(d, n, k, term)
                        for d, n, k in rows]
            self.assertEqual(got, expected, msg=f"term={term}")

## 3 End-to-end on a miniature dump

The fixture below imitates the real files closely: tab-separated lines,
JSON records, `\N` for missing values, and Wikidata JSON wrapped in
CSV-style quotes. It contains deliberate traps, listed next to each
record. The expected values in the tests were worked out by hand from
these records, not by running the code.

| Work | Authors (death year) | Dates | Expected US / PH |
|---|---|---|---|
| W1 | A3 (unknown) | first published 1931 | enters 2027 / unknown |
| W2 | A1 (1976), A2 (1950) | first published 1960, an edition from 1965 | renewal unknown / enters 2027 |
| W3 | A2 (1950) | editions 1890 and 2020 | public domain / public domain |
| W4 | A4 (1970, Filipino) | first published 1955 | renewal unknown / public domain |
| W5 | none | none | unknown / no author |
| W6 | A3 (born 1900) | "first published" 1800 | flagged: year before birth |
| W7 | — | malformed JSON | kept, json_ok = False |

In [4]:
def dump_line(record_type, key, record):
    """One line in the works/editions/authors dump format.

    Real records repeat their key and type inside the JSON; the parser
    relies on that key to tell parsed from malformed records.
    """
    record = {"key": key, "type": {"key": record_type}, **record}
    return "\t".join([record_type, key, "1", "2026-10-01T00:00:00",
                      json.dumps(record)])


def wikidata_line(qid, statements):
    """One line in the wikidata dump format (CSV-quoted JSON)."""
    entity = json.dumps({"id": qid, "type": "item",
                         "statements": statements})
    return "\t".join([qid, '"' + entity.replace('"', '""') + '"',
                      "2026-10-18 00:00:00"])


def time_statement(year, precision=11, rank="normal"):
    return {"rank": rank, "value": {"content": {
        "time": f"+{year:04d}-01-01T00:00:00Z", "precision": precision}}}


def id_statement(value, rank="normal"):
    return {"rank": rank, "value": {"content": value}}


def author_ref(key):
    return {"type": {"key": "/type/author_role"},
            "author": {"key": f"/authors/{key}"}}


works = [
    dump_line("/type/work", "/works/OL1W", {
        "title": "Book Nineteen Thirty-One", "first_publish_date": "1931",
        "authors": [author_ref("OL3A")], "subjects": ["Fiction"]}),
    dump_line("/type/work", "/works/OL2W", {
        "title": "Two Authors", "first_publish_date": "May 1960",
        "authors": [author_ref("OL1A"), author_ref("OL2A")]}),
    dump_line("/type/work", "/works/OL3W", {
        "title": "Old Classic", "authors": [author_ref("OL2A")]}),
    dump_line("/type/work", "/works/OL4W", {
        "title": "Filipino Novel", "first_publish_date": "1955",
        "authors": [author_ref("OL4A")]}),
    dump_line("/type/work", "/works/OL5W", {"title": "No Author"}),
    dump_line("/type/work", "/works/OL6W", {
        "title": "Impossible Date", "first_publish_date": "1800",
        "authors": [author_ref("OL3A")]}),
    "/type/work\t/works/OL7W\t1\t2026-10-01T00:00:00\t{not valid json",
    # a redirect record must be ignored
    dump_line("/type/redirect", "/works/OL8W", {"location": "/works/OL1W"}),
]

editions = [
    dump_line("/type/edition", "/books/OL1M", {
        "works": [{"key": "/works/OL1W"}], "publish_date": "1931",
        "publishers": ["Old Press Ltd."],
        "languages": [{"key": "/languages/eng"}]}),
    dump_line("/type/edition", "/books/OL2M", {
        "works": [{"key": "/works/OL1W"}], "publish_date": "2020",
        "publishers": ["Old Press"], "ocaid": "oldbook00",
        "languages": [{"key": "/languages/tgl"}]}),
    dump_line("/type/edition", "/books/OL3M", {
        "works": [{"key": "/works/OL3W"}], "publish_date": "1890",
        "publishers": ["Alpha"]}),
    dump_line("/type/edition", "/books/OL4M", {
        "works": [{"key": "/works/OL3W"}], "publish_date": "March 2020",
        "publishers": ["Beta Inc."],
        "languages": [{"key": "/languages/fil"}]}),
    dump_line("/type/edition", "/books/OL5M", {
        "works": [{"key": "/works/OL2W"}], "publish_date": "1965"}),
    # an edition with no work, and one pointing to a missing work
    dump_line("/type/edition", "/books/OL6M", {"publish_date": "2019"}),
    dump_line("/type/edition", "/books/OL7M", {
        "works": [{"key": "/works/OL999W"}], "publish_date": "2018"}),
]

authors = [
    dump_line("/type/author", "/authors/OL1A", {
        "name": "Died Seventy-Six", "death_date": "1976"}),
    # Open Library says 1951 but Wikidata says 1950: Wikidata wins
    dump_line("/type/author", "/authors/OL2A", {
        "name": "Mid Century", "death_date": "1951"}),
    dump_line("/type/author", "/authors/OL3A", {
        "name": "Living Or Unknown", "birth_date": "1900"}),
    # linked to Wikidata only through remote_ids
    dump_line("/type/author", "/authors/OL4A", {
        "name": "Filipino Author", "remote_ids": {"wikidata": "Q4"}}),
]

wikidata = [
    wikidata_line("Q2", {
        "P648": [id_statement("OL2A"), id_statement("OL2W")],
        "P570": [time_statement(1950),
                 time_statement(1999, rank="deprecated"),  # ignored
                 time_statement(2000, precision=8)],       # decade only
        "P569": [time_statement(1880)],
    }),
    wikidata_line("Q4", {
        "P570": [time_statement(1970)],
        "P27": [id_statement("Q928")],                     # Philippines
    }),
]

reading_log = [
    "/works/OL1W\t\\N\tWant to Read\t2026-10-01",
    "/works/OL1W\t/books/OL1M\tWant to Read\t2025-12-01",
    "/works/OL1W\t\\N\tWant to Read\t2024-06-01",
    "/works/OL1W\t\\N\tAlready Read\t2026-01-15",
    "/works/OL3W\t\\N\tWant to Read\t2026-01-01",
    "/works/OL2W\t\\N\tCurrently Reading\t2026-02-01",
    "/works/OL1W\t\\N\tSome Custom Shelf\t2026-02-01",      # dropped
]

ratings = [
    "/works/OL1W\t\\N\t4\t2026-01-01",
    "/works/OL1W\t/books/OL2M\t5\t2026-02-01",
    "/works/OL3W\t\\N\t9\t2026-02-01",                       # invalid
]

tmp = tempfile.TemporaryDirectory()
fixture_dir = Path(tmp.name) / "dumps"
fixture_dir.mkdir()
for name, lines in {"works": works, "editions": editions,
                    "authors": authors, "wikidata": wikidata,
                    "reading-log": reading_log, "ratings": ratings}.items():
    path = fixture_dir / f"ol_dump_{name}_2026-10-31.txt.gz"
    with gzip.open(path, "wt", encoding="utf-8") as handle:
        handle.write("\n".join(lines) + "\n")

dumps = helper.discover_dumps(fixture_dir)
paths = helper.build_parquet_tables(spark, dumps, Path(tmp.name) / "pq")
tables = helper.load_tables(spark, paths)
deaths = helper.reconcile_author_deaths(tables["authors"],
                                        tables["wikidata"])
status = helper.estimate_copyright_status(
    tables["works"], tables["editions"], deaths["author_deaths"])
demand = helper.compute_demand(tables["reading_log"], tables["ratings"])
supply = helper.compute_supply(tables["editions"])
profile = helper.build_work_profile(status, demand, supply).cache()
rows = {r["work_key"]: r.asDict()
        for r in profile.collect()}
print("Fixture built:", {k: v.count() for k, v in tables.items()})

Fixture built: {'works': 7, 'editions': 7, 'authors': 4, 'wikidata': 2, 'reading_log': 6, 'ratings': 3}


In [5]:
def work(n):
    return rows[f"/works/OL{n}W"]


class TestParsing(unittest.TestCase):

    def test_record_counts(self):
        # the redirect is dropped; the malformed work is kept
        self.assertEqual(tables["works"].count(), 7)
        self.assertEqual(tables["editions"].count(), 7)
        self.assertEqual(tables["authors"].count(), 4)
        self.assertEqual(tables["wikidata"].count(), 2)
        self.assertEqual(tables["reading_log"].count(), 6)
        self.assertEqual(tables["ratings"].count(), 3)

    def test_malformed_json_is_flagged_not_dropped(self):
        bad = tables["works"].filter("work_key = '/works/OL7W'").collect()
        self.assertEqual(len(bad), 1)
        self.assertFalse(bad[0]["json_ok"])

    def test_profile_counts_malformed_json(self):
        profile_table = helper.profile_tables(tables)
        parsed = profile_table.query(
            "table == 'works' and field == 'json parsed'").iloc[0]
        self.assertEqual((parsed["rows"], parsed["non_null"]), (7, 6))

    def test_invalid_rating_becomes_null(self):
        values = sorted(r["rating"] is None
                        for r in tables["ratings"].collect())
        self.assertEqual(values, [False, False, True])

    def test_parquet_reused_without_raw_dumps(self):
        # once tables exist, the raw dumps are no longer needed
        reused = helper.build_parquet_tables(spark, {}, Path(tmp.name) / "pq")
        self.assertEqual(set(reused), set(helper.DUMP_NAMES))
        # but a missing table with no dump is an error, not a silent skip
        with self.assertRaises(FileNotFoundError):
            helper.build_parquet_tables(spark, {}, Path(tmp.name) / "empty")
        self.assertEqual(
            helper.discover_dumps(Path(tmp.name), required=False), {})

    def test_no_duplicate_keys(self):
        dupes = helper.find_duplicate_keys(tables)
        self.assertEqual(dupes["duplicated_keys"].tolist(), [0, 0, 0])


class TestAuthorDeaths(unittest.TestCase):

    def setUp(self):
        self.by_key = {r["author_key"]: r for r in
                       deaths["author_deaths"].collect()}

    def test_wikidata_wins_and_deprecated_ignored(self):
        a2 = self.by_key["/authors/OL2A"]
        self.assertEqual(a2["death_year"], 1950)
        self.assertEqual(a2["death_source"], "wikidata")
        self.assertEqual(a2["birth_year"], 1880)

    def test_open_library_fallback(self):
        a1 = self.by_key["/authors/OL1A"]
        self.assertEqual((a1["death_year"], a1["death_source"]),
                         (1976, "open_library"))

    def test_link_through_remote_ids(self):
        a4 = self.by_key["/authors/OL4A"]
        self.assertEqual(a4["death_year"], 1970)
        self.assertTrue(a4["wd_philippine_citizen"])

    def test_agreement_table(self):
        agreement = deaths["agreement"].iloc[0]
        # only OL2A has both sources: 1951 vs 1950
        self.assertEqual(agreement["authors_with_both"], 1)
        self.assertEqual(agreement["exact_match"], 0)
        self.assertEqual(agreement["within_one_year"], 1)


class TestStatus(unittest.TestCase):

    def test_first_year(self):
        self.assertEqual(work(1)["first_year"], 1931)
        self.assertEqual(work(2)["first_year"], 1960)   # 1960 < 1965
        self.assertEqual(work(3)["first_year"], 1890)   # from editions
        self.assertEqual(work(3)["year_source"], "earliest edition")
        self.assertIsNone(work(5)["first_year"])

    def test_us_and_ph_codes(self):
        expected = {1: ("enters", "unknown_death"),
                    2: ("renewal", "enters"),
                    3: ("pd", "pd"),
                    4: ("renewal", "pd"),
                    5: ("unknown", "unknown_author")}
        for n, codes in expected.items():
            with self.subTest(work=n):
                self.assertEqual((work(n)["us_code"], work(n)["ph_code"]),
                                 codes)

    def test_last_author_counts(self):
        self.assertEqual(work(2)["n_authors"], 2)
        self.assertEqual(work(2)["last_death_year"], 1976)

    def test_flags(self):
        self.assertTrue(work(6)["year_before_birth"])
        self.assertTrue(work(6)["flagged"])
        self.assertFalse(work(6)["potentially_pd"])
        self.assertTrue(work(1)["enters_2027"])
        self.assertTrue(work(2)["enters_2027"])
        self.assertTrue(work(3)["potentially_pd"])
        self.assertTrue(work(4)["has_philippine_author"])

    def test_status_summary_adds_up(self):
        summary = helper.summarize_status(status)
        self.assertEqual(summary["us"]["works"].sum(), 7)
        self.assertEqual(summary["ph"]["works"].sum(), 7)
        self.assertEqual(int(summary["crosstab"].values.sum()), 7)


class TestDemandAndSupply(unittest.TestCase):

    def test_shelves_and_window(self):
        w1 = work(1)
        self.assertEqual(w1["want_to_read"], 3)
        self.assertEqual(w1["already_read"], 1)
        # window (2025-10-31, 2026-10-31] holds Oct 2026 and Dec 2025
        self.assertEqual(w1["wtr_last_12m"], 2)
        self.assertEqual(w1["wtr_prev_12m"], 0)
        self.assertEqual(work(2)["currently_reading"], 1)

    def test_ratings(self):
        self.assertEqual(work(1)["ratings_count"], 2)
        self.assertAlmostEqual(work(1)["ratings_mean"], 4.5)
        self.assertEqual(work(3)["ratings_count"], 0)   # 9 is invalid

    def test_supply_counts(self):
        w1 = work(1)
        self.assertEqual(w1["n_editions"], 2)
        self.assertEqual(w1["n_recent_editions"], 1)
        self.assertEqual(w1["n_publishers"], 1)      # "Old Press" twice
        self.assertEqual(w1["n_recent_publishers"], 1)
        self.assertEqual(w1["n_digitized"], 1)
        self.assertEqual(w1["n_filipino_editions"], 1)
        self.assertEqual(work(3)["n_filipino_editions"], 1)  # fil code
        self.assertEqual(work(3)["n_publishers"], 2)
        self.assertEqual(work(5)["n_editions"], 0)   # zero, not missing


class TestWatchlist(unittest.TestCase):

    def test_watchlist_selection(self):
        listed = helper.build_watchlist(profile, min_want_to_read=1,
                                        max_recent_editions=1, top_n=10)
        # W2 also enters 2027 but nobody wants to read it yet
        self.assertEqual(listed["work_key"].tolist(), ["/works/OL1W"])

    def test_author_names_in_credit_order(self):
        listed = helper.build_pd_list(profile, min_want_to_read=0,
                                      top_n=10, authors=tables["authors"])
        names = dict(zip(listed["work_key"], listed["authors"]))
        self.assertEqual(names["/works/OL3W"], "Mid Century")
        listed = helper.build_watchlist(profile, 0, 5, top_n=10,
                                        authors=tables["authors"])
        names = dict(zip(listed["work_key"], listed["authors"]))
        self.assertEqual(names["/works/OL2W"],
                         "Died Seventy-Six, Mid Century")

    def test_low_supply_cutoff(self):
        listed = helper.build_watchlist(profile, min_want_to_read=1,
                                        max_recent_editions=0, top_n=10)
        self.assertTrue(listed.empty)   # W1 has one recent edition

    def test_pd_list(self):
        listed = helper.build_pd_list(profile, min_want_to_read=1, top_n=10)
        self.assertEqual(listed["work_key"].tolist(), ["/works/OL3W"])

    def test_low_supply_pd_list(self):
        listed = helper.build_pd_list(profile, min_want_to_read=1, top_n=10,
                                      max_recent_editions=0)
        self.assertTrue(listed.empty)   # W3 has one edition from 2020

    def test_top_authors(self):
        table = helper.top_authors(profile, tables["authors"],
                                   "enters_2027", top_n=10)
        first = table.iloc[0]
        self.assertEqual((first["name"], first["works"],
                          first["want_to_read"], first["via_us_1931"]),
                         ("Living Or Unknown", 1, 3, 1))
        # W2 counts once for each of its two authors, both via PH 1976
        rest = table[table["name"] != "Living Or Unknown"]
        self.assertEqual(sorted(rest["name"]),
                         ["Died Seventy-Six", "Mid Century"])
        self.assertEqual(rest["via_ph_1976"].tolist(), [1, 1])

    def test_group_comparison(self):
        groups = helper.compare_groups(profile).set_index("us_code")
        self.assertEqual(groups["works"].sum(), 7)
        self.assertEqual(groups.loc["enters", "total_wtr"], 3)
        self.assertEqual(groups.loc["enters", "wtr_last_12m"], 2)
        # no entries in the earlier window, so growth is undefined
        self.assertTrue(pd.isna(groups.loc["enters", "wtr_growth_pct"]))
        self.assertAlmostEqual(groups.loc["enters", "mean_rating"], 4.5)

    def test_flag_summary(self):
        table = helper.list_flag_summary(profile, 1).set_index("list")
        self.assertEqual(
            table.loc["potentially public domain, high demand", "works"], 1)
        self.assertEqual(table.loc["enters 2027, high demand", "works"], 1)
        self.assertEqual(table["gap_after_death"].sum(), 0)

    def test_threshold_ignores_unshelved_works(self):
        # WTR counts among works with any entry are [3, 1]
        self.assertEqual(helper.demand_threshold(profile, 0.99), 3)

    def test_validation_checks_pass(self):
        listed = helper.build_watchlist(profile, 1, 1, top_n=10)
        checks = helper.validate_analysis(tables, status, profile, listed,
                                          min_want_to_read=1)
        failed = checks[~checks["passed"]]
        self.assertTrue(failed.empty, msg=failed.to_string())

## 4 Code style

`pycodestyle` checks line length, spacing and other PEP 8 conventions,
so the module stays easy to read for the peer reviewers.

In [6]:
class TestStyle(unittest.TestCase):

    def test_pep8(self):
        import pycodestyle

        report = pycodestyle.StyleGuide(quiet=True).check_files(
            ["helper_code.py"])
        self.assertEqual(report.total_errors, 0)

## 5 Run every test

In [7]:
suite = unittest.TestSuite()
loader = unittest.defaultTestLoader
for case in [TestYearParsing, TestPublisherNames, TestCopyrightRules,
             TestSparkTwins, TestParsing, TestAuthorDeaths, TestStatus,
             TestDemandAndSupply, TestWatchlist, TestStyle]:
    suite.addTests(loader.loadTestsFromTestCase(case))
# write the per-test log to stdout so it is kept with the notebook
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
print(f"\n{result.testsRun - len(result.failures) - len(result.errors)}"
      f" of {result.testsRun} tests passed")
tmp.cleanup()
assert result.wasSuccessful(), "Some tests failed; see the log above."

test_life_years (__main__.TestYearParsing.test_life_years) ... 

ok


test_publication_years (__main__.TestYearParsing.test_publication_years) ... 

ok


test_wikidata_time (__main__.TestYearParsing.test_wikidata_time) ... 

ok


... (78 outputs left out)

test_pep8 (__main__.TestStyle.test_pep8) ... 

ok


----------------------------------------------------------------------
Ran 43 tests in 15.039s

OK



43 of 43 tests passed
